# 10｜YOLOv1 的预测张量：一个网格位置到底输出什么

上一课认识了 Backbone、Neck 和 Detection Head。现在我们把注意力集中到网络最后：**检测头的输出究竟长什么样？**

这一节参考“薛什么的B站”课程中“6.2 YOLOv1 的预测张量”的学习顺序，并对照 YOLOv1 原论文重新组织为当前笔记的表述。这里只学习预测张量；坐标换算、损失和 NMS 分别留到后面。

## 一、先从最终答案倒推

一条目标检测结果至少要回答：

1. 物体在哪里：边界框；
2. 模型有多确定：置信度；
3. 物体是什么：类别。

YOLOv1 不会只输出一条结果。它先把图片划分成网格，让每个网格位置都有机会提出预测，最后把所有位置的输出整理成一个张量。

因此，预测张量可以先理解为：

**整张图片中所有预测槽位的答案表。**

## 二、YOLOv1 先把图片划分成网格

YOLOv1 把图片划分成 $S\times S$ 个网格位置：

$$
\text{图片}
\longrightarrow
S\times S\text{ 个网格位置}
$$

如果 $S=7$，就有：

$$
7\times7=49
$$

个网格位置。

这里的网格不是把图片切开后分别送进 49 个独立模型。整张图片仍然一起经过神经网络，网络能够利用全局信息；网格只是组织最终预测的一种空间结构。

## 三、哪个网格位置负责一个真实物体

YOLOv1 使用一条很直观的责任规则：

**真实物体的中心点落在哪个网格位置，就由哪个位置负责这个物体。**

注意，是看物体边界框的中心点，不是看物体的大部分面积落在哪里。

一只大狗可能跨越许多网格，但它的中心点只落在其中一个位置。负责位置会预测整只狗的边界框，框可以远远超出该网格范围。

所以：

- 网格位置决定谁负责；
- 边界框输出决定物体实际覆盖多大范围。

## 四、每个网格位置不只预测一个框

在 YOLOv1 中，每个网格位置会预测 $B$ 个候选边界框。

每个候选框包含 5 个数：

$$
(x,y,w,h,\operatorname{confidence})
$$

其中：

- $x,y$：框中心的位置；
- $w,h$：框的宽和高；
- confidence：这个框的置信度。

因此，$B$ 个候选框一共需要：

$$
5B
$$

个输出数字。

为什么一个位置准备多个框？因为不同框预测器可以逐渐擅长不同形状或大小的目标，也给模型提供了多个定位方案。具体哪个框在训练时负责真实目标，后面学习 IoU 与损失时再讲。

## 五、每个网格位置还要预测类别

假设数据集中共有 $C$ 个类别，每个网格位置还会输出 $C$ 个类别概率：

$$
\bigl(
P(c_1\mid\text{有物体}),
\ldots,
P(c_C\mid\text{有物体})
\bigr)
$$

它们回答：如果这个网格位置负责的确有物体，那么它更可能属于哪个类别？

YOLOv1 有一个非常重要的特点：

**一个网格位置只输出一组 $C$ 个类别概率，而不是为每个候选框各输出一组类别概率。**

因此不能把每个框都算成 $5+C$ 个数。YOLOv1 每个位置的结构是 $B$ 组框参数，再加一组共享的类别概率。

## 六、得到预测张量的核心公式

每个网格位置输出：

$$
\underbrace{5B}_{B\text{ 个候选框}}
+
\underbrace{C}_{\text{一组类别概率}}
$$

整张图片共有 $S\times S$ 个位置，所以单张图片的预测张量形状是：

$$
S\times S\times(5B+C)
$$

这就是 YOLOv1 预测张量最核心的公式。

三个维度分别表示：

- 第一个 $S$：网格行；
- 第二个 $S$：网格列；
- $5B+C$：这个位置需要说出的全部检测信息。

## 七、先用一个很小的例子

假设为了教学，我们设置：

- $S=2$：图片划分为 $2\times2$；
- $B=2$：每个位置预测 2 个候选框；
- $C=3$：类别只有猫、狗、汽车。

每个位置的输出长度为：

$$
5B+C=5\times2+3=13
$$

整张图片的预测张量就是：

$$
2\times2\times13
$$

其中任意一个网格位置的 13 个数字可以拆成：

| 部分 | 数量 | 内容 |
|---|---:|---|
| 候选框 1 | 5 | $x_1,y_1,w_1,h_1,\operatorname{conf}_1$ |
| 候选框 2 | 5 | $x_2,y_2,w_2,h_2,\operatorname{conf}_2$ |
| 类别概率 | 3 | 猫、狗、汽车 |

## 八、再看 YOLOv1 论文中的真实设置

YOLOv1 在 PASCAL VOC 上使用：

$$
S=7,\qquad B=2,\qquad C=20
$$

因此每个位置输出：

$$
5\times2+20=30
$$

个数字，单张图片最终得到：

$$
7\times7\times30
$$

的预测张量。

这里的 30 不是 30 个物体，也不是 30 个类别，而是一个网格位置需要输出的全部数字数量。

## 九、为什么一张图只有 98 个候选框

YOLOv1 有 $7\times7=49$ 个网格位置，每个位置预测 2 个框，所以候选框数量为：

$$
7\times7\times2=98
$$

预测张量虽然是 $7\times7\times30$，但不能把所有元素相乘后说成 1470 个框，因为最后一个维度还混合了框参数和类别概率。

正确的拆法是：

$$
\underbrace{7\times7\times2}_{98\text{ 个候选框}}
\quad+\quad
\underbrace{7\times7\times20}_{49\text{ 组类别概率}}
$$

98 也只是候选框数量，不代表图片中存在 98 个真实物体。

## 十、预测张量和特征图是什么关系

在进入检测头之前，网络内部保存的是特征图；经过最后的预测层以后，才得到预测张量：

$$
\text{图片}
\longrightarrow
\text{卷积网络提取特征}
\longrightarrow
\text{内部特征图}
\longrightarrow
\text{最终预测层}
\longrightarrow
7\times7\times30
$$

特征图中的通道表示网络学到的内部特征；预测张量最后的 30 个通道具有明确任务含义，例如框坐标、置信度和类别概率。

因此：

- 特征图是模型思考时使用的信息；
- 预测张量是模型把思考结果整理成的检测答案。

## 十一、一个网格位置的输出槽位怎样理解

取第 $i$ 行、第 $j$ 列的位置，它对应一个长度为 $5B+C$ 的向量：

$$
\mathbf{y}_{ij}\in\mathbb{R}^{5B+C}
$$

在 YOLOv1 的 VOC 设置中：

$$
\mathbf{y}_{ij}\in\mathbb{R}^{30}
$$

这个向量就是该网格位置的输出内容。把 49 个位置的向量按照行列排列起来，就形成 $7\times7\times30$ 的预测张量。

所以预测槽位不是一个现实中的盒子。它只是张量中的固定位置，例如“第 3 行、第 5 列的第一个框预测器”。

## 十二、不要把两个字母 B 混在一起

在 YOLOv1 公式中，$B$ 表示**每个网格位置预测多少个候选框**。

但在 PyTorch 学习中，我们经常也用 $B$ 表示 batch size。为了避免混淆，这里把 batch size 写成 $N$。

单张图片：

$$
S\times S\times(5B+C)
$$

一个 batch：

$$
N\times S\times S\times(5B+C)
$$

前面的 $N$ 是图片数量，括号里的 $B$ 是每个网格位置的候选框数量，两者没有关系。

## 十三、类别分数怎样与候选框结合

每个网格位置只有一组类别概率，但有 $B$ 个候选框置信度。推理时，可以把某类别概率与某个框的置信度相乘，得到这个框针对该类别的分数：

$$
\operatorname{score}_{i,b,c}
=
P(c\mid\text{有物体})
\times
\operatorname{confidence}_{i,b}
$$

可以用人话理解：

- 类别概率回答“如果这里有物体，它像不像狗”；
- 框置信度回答“这个候选框本身有多可信”；
- 两者结合后，才得到“这个框作为狗框有多可信”。

下一节学习坐标换算后，我们才能把这些分数与真正画在原图上的框联系起来。

## 十四、训练时真实标注怎样进入这张表

模型前向传播产生预测张量；训练数据提供真实类别和真实边界框。目标分配规则会确定哪个网格位置负责哪个目标，并为需要监督的输出部分准备目标值。

$$
\begin{aligned}
\text{图片}&\longrightarrow\text{模型预测张量}\\
\text{真实标注}&\longrightarrow\text{对应位置的监督目标}
\end{aligned}
\quad\Longrightarrow\quad
\text{计算损失}
$$

没有物体中心落入的位置主要学习降低置信度；负责物体的位置还要学习类别和边界框。

因此，预测张量是网络算出来的，监督目标是根据人工标注和分配规则构造的，两者不能混为一谈。

## 十五、YOLOv1 网格设计的一个局限

一个网格位置只有一组类别概率，并且由物体中心决定责任。

如果两个物体的中心恰好落在同一个网格位置，尤其是两个很靠近的小物体，YOLOv1 就很难让这个位置同时表达两个不同类别的目标。

这也是原始 YOLOv1 对密集小目标处理不够理想的原因之一。后续 YOLO 版本改变了网络结构、预测方式和多尺度设计，但我们先理解 YOLOv1，因为它最容易看清“网格位置怎样组织预测”的基本思想。

## 十六、常见误区

### 误区 1：$7\times7$ 表示把 448 像素图片切成小图分别识别

不是。整张图片一起经过网络，$7\times7$ 是组织最终输出的空间网格。

### 误区 2：一个物体跨越多个格子，就由多个格子共同负责

YOLOv1 按物体中心点决定负责位置。

### 误区 3：每个位置的两个框各自拥有 20 个类别概率

不是。YOLOv1 的两个框各有 5 个框相关输出，但整个网格位置共享一组 20 类概率。

### 误区 4：$7\times7\times30$ 表示 1470 个候选框

不是。候选框数量是 $7\times7\times2=98$。

### 误区 5：预测张量就是 Backbone 输出的普通特征图

不是。普通特征图保存内部特征；最终预测张量的各部分已经具有明确检测含义。

## 十七、自我检查

1. YOLOv1 用什么规则决定哪个网格位置负责一个物体？
2. 每个候选框为什么包含 5 个数？
3. 为什么每个网格位置的输出长度是 $5B+C$，而不是 $B(5+C)$？
4. 当 $S=7$、$B=2$、$C=20$ 时，预测张量是什么形状？
5. YOLOv1 一张图片会产生多少个候选框？
6. 特征图和最终预测张量有什么区别？
7. batch size 和 YOLOv1 公式中的 $B$ 有什么区别？

### 参考答案

1. 物体边界框的中心点落在哪个网格位置，就由哪个位置负责。
2. 4 个数描述中心与宽高，第 5 个数是框的置信度。
3. 因为 $B$ 个框分别输出 5 个框相关数值，但每个网格位置只共享一组 $C$ 个类别概率。
4. $7\times7\times30$。
5. $7\times7\times2=98$ 个候选框。
6. 特征图保存网络内部提取的信息；预测张量保存具有明确含义的框、置信度和类别输出。
7. batch size 是一次处理的图片数；YOLOv1 公式中的 $B$ 是每个网格位置预测的候选框数。

## 十八、本课总结

这节课最核心的拆分是：

$$
\underbrace{S\times S}_{\text{多少个空间位置}}
\times
\underbrace{(5B+C)}_{\text{每个位置说多少个数字}}
$$

对于 YOLOv1 的 VOC 设置：

$$
7\times7\times(5\times2+20)
=
7\times7\times30
$$

它表示 49 个网格位置，每个位置准备 2 个候选框并共享一组 20 类概率，整张图片共提出 98 个候选框。

## 下一课：YOLOv1 的坐标换算

现在我们知道每个框输出 $x,y,w,h$，但还不知道这些数字相对于谁。

下一课会用一只狗和一个 $7\times7$ 网格，慢慢计算：

- $x,y$ 为什么相对于负责网格位置；
- $w,h$ 为什么相对于整张图片；
- 怎样把归一化输出还原成原图像素坐标；
- 为什么框可以超出负责它的网格。

## 参考来源

- [薛什么的B站：《零基础入门深度学习》课程目录](https://www.bilibili.com/cheese/play/ss794934934)：本课参考其“6.2 YOLOv1 的预测张量”的主题顺序，笔记内容为重新组织后的原创讲解。
- [YOLOv1 原论文：You Only Look Once: Unified, Real-Time Object Detection](https://arxiv.org/abs/1506.02640)：网格责任规则、每格 $B$ 个框、$C$ 个条件类别概率，以及 $S\times S\times(5B+C)$ 输出定义的主要依据。